# Multi-dataset loading with the current scdata API

This notebook shows the current recommended path for training from multiple
single-cell stores:

1. Write or collect scdata-readable zarr stores (`.zarr.zip` or `.zarr`).
2. Use `Corpus(paths, gene_alignment=...)` for the high-level path.
3. Build a `ScDataLoader` whose batches are decoded by the Rust `ScDataBank`.
4. Consume ordinary torch tensors plus `file_ids` / `cell_ids` metadata.

The important current behavior is that **one torch batch may contain cells from
multiple datasets**. Python passes the batch as `(dataset_idx, cells)` parts;
the Rust databank plans and decodes the whole multi-dataset batch natively, then
returns one `CellBatch` in original row order. `ScDataLoader` exposes that as
`batch["batch"]`; grouped views in `batch["batches"]` remain available for code
that needs per-file slices.

## Current data flow

```
paths -> Corpus -> ScDataBank.register(...) -> DatasetId handles
                                  |
Torch sampler -> (file_id, cell_id) samples
                                  |
ScDataLoader -> bank.prefetch_multi([DatasetId, ...], multi-dataset batches)
                                  |
Rust scheduled prefetch -> one decoded CellBatch per torch batch
                                  |
default collate -> {"x": torch.Tensor, "file_ids", "cell_ids", "gene_names"}
```

`Corpus` is the best default for training. Use the lower-level
`ScDataBank + ScDataLoader` form when you need custom registration, custom
sampling, or direct access to `bank.load()` / `bank.prefetch()` (single dataset)
or `bank.load_multi()` / `bank.prefetch_multi()` (cross-dataset batches).

## Imports

`anndata`, `pandas`, and `scipy` are used only to synthesize small example
stores. Real training code usually starts from existing `.zarr.zip` paths.

In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch
from scipy import sparse

from scdata import (
    CellIndexDataset,
    Corpus,
    DataBankConfig,
    ScDataBank,
    ScDataLoader,
    ScheduledPrefetchConfig,
    __version__,
    kernel_name,
    kernel_version,
    launch,
    write_zarr,
)

print("scdata", __version__, "/", kernel_name(), kernel_version())
print("torch", torch.__version__)

## Create two example stores

The two files intentionally have different gene spaces:

- `sample_a`: sparse CSR, genes `Gene000` to `Gene063`
- `sample_b`: dense, genes `Gene064` to `Gene127`

That lets us demonstrate `gene_alignment="union"`, where the loader projects
every dataset onto one shared gene list and fills missing genes with zeros.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "outputs" / "_nb_multi_dataset_current"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def make_adata(n_cells: int, genes: list[str], *, sparse_x: bool, seed: int) -> ad.AnnData:
    rng = np.random.default_rng(seed)
    if sparse_x:
        dense = rng.poisson(0.15, size=(n_cells, len(genes))).astype(np.float32)
        dense[dense < 1] = 0.0
        X = sparse.csr_matrix(dense)
    else:
        X = (rng.random((n_cells, len(genes)), dtype=np.float32) * 10.0).astype(np.float32)
    obs = pd.DataFrame(index=[f"cell{seed}_{i:04d}" for i in range(n_cells)])
    var = pd.DataFrame(index=genes)
    return ad.AnnData(X=X, obs=obs, var=var)


genes_a = [f"Gene{i:03d}" for i in range(64)]
genes_b = [f"Gene{i:03d}" for i in range(64, 128)]
adata_a = make_adata(320, genes_a, sparse_x=True, seed=1)
adata_b = make_adata(240, genes_b, sparse_x=False, seed=2)

store_paths = [
    write_zarr(
        adata_a,
        DATA_DIR / "sample_a.zarr.zip",
        format="sparse",
        chunk_size=64,
        align_cells=True,
        store="zip",
    ),
    write_zarr(
        adata_b,
        DATA_DIR / "sample_b.zarr.zip",
        format="dense1d",
        chunk_size=64 * len(genes_b),
        align_cells=True,
        store="zip",
    ),
]

store_paths

## Recommended path: `Corpus`

`Corpus` owns a bank when constructed from paths. It launches and registers all
stores, chooses a shared gene axis, and builds loaders with a default dense
collate.

Useful `gene_alignment` modes:

- `"strict"`: all datasets must already have identical gene names and order.
- `"union"`: first-seen union of all genes; missing genes are zero-filled by default.
- `"intersection"`: shared genes only; may be an empty set.
- `"none"`: no projection; use only when mixed batches are already compatible.

In [ ]:
prefetch_config = ScheduledPrefetchConfig.make(
    prefetch_step=2,
    access__decode_ahead_steps=1,
    access__ready_ahead_steps=1,
)

with Corpus(store_paths, gene_alignment="union") as corpus:
    print(corpus)
    print("genes:", corpus.num_genes, corpus.gene_names[:5], "...", corpus.gene_names[-5:])
    print("cells per file:", corpus.cells_per_file)
    print("config summary:", corpus.bank_config_summary)

    loader = corpus.loader(
        batch_size=128,
        shuffle=True,
        drop_last=False,
        prefetch_config=prefetch_config,
        collect_stats=True,
    )

    batch = next(iter(loader))
    print("batch keys:", sorted(batch))
    print("x:", tuple(batch["x"].shape), batch["x"].dtype)
    print("file_ids[:12]:", batch["file_ids"][:12].tolist())
    print("cell_ids[:12]:", batch["cell_ids"][:12].tolist())
    print("gene_names:", len(batch["gene_names"]))

    stats = loader.stats(reset=False)
    print(
        f"stats: batches={stats.batches_seen}, cells={stats.cells_seen}, "
        f"p99_wait={stats.wait_p99_ms:.3f} ms, "
        f"cells/s={stats.throughput_cells_per_s:.0f}"
    )

## Direct access through the same corpus bank

`bank.load_multi()` supports the same multi-dataset semantics as
`prefetch_multi`: pass the list of dataset ids and one batch of
`(dataset_idx, cells)` parts. The result is a `CellData` object whose rows
follow the input part order.

In [ ]:
with Corpus(store_paths, gene_alignment="union") as corpus:
    rows = corpus.bank.load_multi(
        corpus.dataset_ids,
        [(0, [0, 1, 2]), (1, [0, 5])],
        genes=corpus.gene_names,
        missing=corpus.missing,
        dtype="f32",
    )
    print("cells:", rows.cells.tolist())
    print("matrix:", rows.to_numpy().shape, rows.data.dtype)
    print("genes:", len(rows.var_names), rows.var_names[:4])

## Lower-level path: explicit bank + loader

Use this form when you want to control registration or use a custom
`collate_fn`. The current `ScDataLoader` batch dictionary contains:

| key | meaning |
|---|---|
| `batch` | one decoded `CellBatch` for the entire mixed-dataset batch, in original row order |
| `file_ids` | source file id per row |
| `cell_ids` | source cell id per row |
| `batches` | lazy per-file `CellBatch` views, for compatibility or per-file logic |
| `cells` / `positions` | per-file requested cells and positions in the mixed batch |

For most training code, consume `batch["batch"].to_numpy()` directly.

In [ ]:
config = DataBankConfig.make(
    backend="threaded",
    decode__num_workers=4,
    cache_capacity_bytes=128 * 1024**2,
)


def collate_from_single_decoded_batch(batch):
    decoded = batch["batch"]
    return {
        "x": torch.from_numpy(decoded.to_numpy()),
        "file_ids": torch.as_tensor(batch["file_ids"], dtype=torch.long),
        "cell_ids": torch.as_tensor(batch["cell_ids"], dtype=torch.long),
        "gene_names": decoded.var_names,
    }


with ScDataBank(config) as bank:
    dataset_ids = bank.register(launch(path) for path in store_paths)
    cells_per_file = [bank.dataset_num_cells(did) for did in dataset_ids]
    cell_dataset = CellIndexDataset(cells_per_file)

    # Use a subset from the first dataset. Genes absent from the second dataset
    # become zeros because missing="zero".
    genes = bank.dataset_genes(dataset_ids[0])[:16]

    # A deterministic mixed batch: rows alternate file 0 and file 1.
    offset_1 = cells_per_file[0]
    mixed_batch_sampler = [[0, offset_1, 1, offset_1 + 1, 2, offset_1 + 2]]

    loader = ScDataLoader(
        bank,
        cell_dataset,
        batch_sampler=mixed_batch_sampler,
        dataset_ids=dataset_ids,
        genes=genes,
        missing="zero",
        out_dtype="f32",
        prefetch_config=prefetch_config,
        collate_fn=collate_from_single_decoded_batch,
        collect_stats=True,
    )

    batch = next(iter(loader))
    print("x:", tuple(batch["x"].shape), batch["x"].dtype)
    print("file_ids:", batch["file_ids"].tolist())
    print("cell_ids:", batch["cell_ids"].tolist())
    print("genes:", batch["gene_names"])

    stats = loader.stats(reset=False)
    print("loader stats:", stats.batches_seen, "batch,", stats.cells_seen, "cells")

## Inspect the raw `ScDataLoader` batch dictionary

If your `collate_fn` wants per-file views, use `batch["batches"]`. These are
lazy slices derived from the one decoded mixed `CellBatch`; they do not change
what Rust decoded.

In [ ]:
def debug_collate(batch):
    decoded = batch["batch"]
    print("decoded mixed batch:", decoded.shape, decoded.var_names[:3])
    print("row order file_ids:", batch["file_ids"].tolist())
    for file_id, view in batch["batches"].items():
        print(
            "per-file view",
            file_id,
            "cells=",
            batch["cells"][file_id].tolist(),
            "positions=",
            batch["positions"][file_id].tolist(),
            "shape=",
            view.shape,
        )
    return batch


with ScDataBank() as bank:
    dataset_ids = bank.register(launch(path) for path in store_paths)
    cells_per_file = [bank.dataset_num_cells(did) for did in dataset_ids]
    offset_1 = cells_per_file[0]
    loader = ScDataLoader(
        bank,
        CellIndexDataset(cells_per_file),
        batch_sampler=[[0, offset_1, 1, offset_1 + 1]],
        dataset_ids=dataset_ids,
        genes=bank.dataset_genes(dataset_ids[0])[:8],
        missing="zero",
        collate_fn=debug_collate,
    )
    _ = next(iter(loader))

## Existing `.h5ad` / `.loom` / `.mtx` inputs

The example above wrote scdata stores directly with `write_zarr`. For existing
AnnData-readable files, convert them first with `AnnDataZarrZipConverter`, then
feed the returned paths into `Corpus` or `ScDataLoader.from_paths`.

In [ ]:
# Optional conversion pattern for real inputs:
#
# from scdata import AnnDataZarrZipConverter
# converter = AnnDataZarrZipConverter(
#     chunk_size=100_000,
#     align_cells=True,
#     layer_format="auto",
#     overwrite=True,
# )
# store_paths = [converter(path) for path in raw_paths]
#
# with Corpus(store_paths, gene_alignment="union") as corpus:
#     loader = corpus.loader(batch_size=1024, shuffle=True)

## Empty intersections are valid

If `gene_alignment="intersection"` finds no shared genes, scdata returns batches
with shape `[batch_size, 0]`. This is a valid zero-column projection and is
handled by `CellData`, `CellBatch`, and the default collate.

In [ ]:
with Corpus(store_paths, gene_alignment="intersection") as corpus:
    print("intersection genes:", len(corpus.gene_names), corpus.gene_names[:5])
    loader = corpus.loader(batch_size=64, shuffle=False)
    batch = next(iter(loader))
    print("x shape:", tuple(batch["x"].shape))

## Lifecycle notes

- Prefer `with Corpus(...)` or `with ScDataBank(...)` so Rust thread pools and
  file handles are released deterministically.
- `ScDataLoader` does not use torch worker processes. Keep `num_workers=0`; the
  Rust bank owns IO, decode, access, and compute pools.
- `ScheduledPrefetchConfig` is per-loader/per-call. You can try different
  prefetch depths without rebuilding the bank.
- `DataBankConfig` is bank-level. Create a new `ScDataBank` to change IO/cache
  pool sizes.

## Recap

- Use `Corpus(paths, gene_alignment="union" | "strict" | "intersection")` for
  the usual multi-file training path.
- `ScDataLoader` treats the Rust-decoded mixed batch as the primary output:
  `batch["batch"]`.
- `stitch_dense_collate` is the default `Corpus.loader()` collate and returns a
  dense tensor under `batch["x"]`.
- Lower-level `ScDataBank.load()` / `bank.prefetch()` serve a single dataset;
  `ScDataBank.load_multi()` / `bank.prefetch_multi()` accept multiple dataset
  ids and `(dataset_idx, cells)` batch parts directly.
- Missing genes are controlled by `missing="zero"` or `missing="error"` when a
  projection gene list is supplied.